# Ejemplo: Relación entre magnitud y flujo en astronomía

**Ecuación base:**
$$m_1 - m_2 = -2.5 \log_{10}\left(\frac{F_1}{F_2}\right)$$

**Datos:**
- $m_1 < m_2$ → la estrella 1 es más brillante.
- $m_2 - m_1 = 5$ → diferencia de magnitud de 5.

**Desarrollo:**

1. De $m_2 - m_1 = 5$, se tiene $m_1 - m_2 = -5$.

2. Sustituyendo en la ecuación:
   $$-5 = -2.5 \log_{10}\left(\frac{F_1}{F_2}\right)$$

3. Dividiendo ambos lados por $-2.5$:
   $$2 = \log_{10}\left(\frac{F_1}{F_2}\right)$$

4. Aplicando la función exponencial base 10:
   $$10^2 = \frac{F_1}{F_2} \quad \Rightarrow \quad \frac{F_1}{F_2} = 100$$

5. Por lo tanto:
   $$F_1 = 100 \, F_2$$

**Conclusión:**  
Una diferencia de 5 magnitudes corresponde exactamente a un factor de 100 en flujo. Esto es la base de la escala logarítmica de magnitudes: cada magnitud representa un factor de $100^{1/5} \approx 2.512$ en brillo.

**Fórmula:** $F = \dfrac{L}{4\pi r^2}$

**Significado físico:** Ley del inverso del cuadrado para la radiación.

**Términos:**
| Símbolo | Significado | Unidades típicas |
|---------|-------------|-----------------|
| $F$ | Flujo recibido (energía por área por tiempo) | $\text{W/m}^2$ |
| $L$ | Luminosidad intrínseca de la fuente | $\text{W}$ |
| $r$ | Distancia entre fuente y observador | $\text{m}$ |

**Explicación concisa:**  
La energía total $L$ emitida por una fuente se distribuye uniformemente sobre la superficie de una esfera de radio $r$, cuya área es $4\pi r^2$. Por tanto, el flujo $F$ que llega a un detector disminuye con el cuadrado de la distancia.

**Aplicación clave:**  
Permite calcular distancias astronómicas si se conoce $L$ (ej. candelas estándar) o determinar $L$ midiendo $F$ y $r$.

# Análisis de datos

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Datos experimentales (distancia en cm, flujo en lúmenes)
distancias_cm = np.array([10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100])
medicion_1 = np.array([657, 308, 260, 171, 141, 106, 89, 72, 58, 52, 49, 39, 37, 29, 29, 27, 24, 22, 18])
medicion_2 = np.array([576, 315, 229, 167, 131, 111, 94, 72, 66, 57, 51, 44, 39, 38, 37, 30, 28, 26, 23])
medicion_3 = np.array([715, 400, 285, 212, 155, 130, 102, 90, 83, 58, 49, 45, 39, 36, 32, 29, 26, 26, 23])
medicion_4 = np.array([650, 450, 435, 340, 253, 210, 196, 153, 136, 108, 98, 86, 77, 68, 70, 57, 58, 54, 43])

# Promedio de las 4 mediciones + desviación estándar
flujo_promedio = np.mean([medicion_1, medicion_2, medicion_3, medicion_4], axis=0)
flujo_std = np.std([medicion_1, medicion_2, medicion_3, medicion_4], axis=0)

# Convertir distancia a metros para el modelo físico
distancias_m = distancias_cm / 100.0

# Modelo teórico: F = L / (4πr²)
def modelo_inverso_cuadrado(r, L):
    return L / (4 * np.pi * r**2)

# Ajuste de L usando curve_fit (ponderado por incertidumbre)
popt, pcov = curve_fit(modelo_inverso_cuadrado, distancias_m, flujo_promedio, 
                       sigma=flujo_std, absolute_sigma=True, p0=[1.0])
L_ajustado = popt[0]
print(f"Luminosidad ajustada: L = {L_ajustado:.3f} W·sr (unidades arbitrarias)")

# Generar curva teórica suave
r_suave = np.linspace(0.08, 1.05, 200)  # de 8 cm a 105 cm en metros
F_teorico = modelo_inverso_cuadrado(r_suave, L_ajustado)

# Plot
plt.figure(figsize=(10, 6))

# Datos experimentales individuales (con transparencia)
plt.plot(distancias_cm, medicion_1, 'o', color='blue', alpha=0.4, label='Medición 1', markersize=4)
plt.plot(distancias_cm, medicion_2, 's', color='green', alpha=0.4, label='Medición 2', markersize=4)
plt.plot(distancias_cm, medicion_3, '^', color='orange', alpha=0.4, label='Medición 3', markersize=4)
plt.plot(distancias_cm, medicion_4, 'd', color='purple', alpha=0.4, label='Medición 4', markersize=4)

# Promedio con barras de error
plt.errorbar(distancias_cm, flujo_promedio, yerr=flujo_std, fmt='none', 
             ecolor='red', capsize=3, label='Promedio ± σ')

# Curva teórica ajustada
plt.plot(r_suave*100, F_teorico, 'k-', linewidth=2, label=f'Teoría: $F=L/(4\\pi r^2)$\n$L$={L_ajustado:.1f}')

# Formato
plt.title('Flujo luminoso vs Distancia: Datos experimentales y ley del inverso del cuadrado')
plt.xlabel('Distancia (cm)')
plt.ylabel('Flujo (lúmenes)')
plt.legend(fontsize=9, ncol=2)
plt.grid(True, alpha=0.3, linestyle='--')
#plt.yscale('log')  # Escala logarítmica para visualizar mejor el decaimiento
#plt.xscale('log')
plt.tight_layout()
plt.show()

# Opcional: guardar figura
# plt.savefig('astroposicion_flujo_vs_distancia.png', dpi=300)